# Week 8 Capstone — Meeting-to-Action Assistant

This notebook starts from the Week 8 capstone skeleton and turns it into the final integrated system. It accepts an existing meeting transcript and produces a structured operational brief with an executive summary, decisions, action items and owners, blockers, and follow-up questions.

## Rubric map

| Criterion | Evidence in this project |
| --- | --- |
| Integration by Reuse | Week 3 structured prompting, Week 4 validation, Week 5 MiniLM/FAISS RAG, Week 6 cross-encoder reranking, Week 7 dynamic few-shot retrieval |
| Evaluation | 12 held-out cases, deterministic metrics, latency, structured validity, plus Gemini judge on correctness, grounding, and operational usefulness |
| Required Failure Mode | A/B/C ablation, retrieval logs, automatic regression trace, and deterministic mitigation |
| README and Architecture | `project/README.md` documents the pipeline and rationale |
| Portfolio Curation | polished project README, eight-week course history, this notebook, and `project/research-note.md` |

### Reused course components

1. **Week 3:** structured prompt/output discipline.
2. **Week 4:** deterministic schema validation and structured failure handling.
3. **Week 5:** `all-MiniLM-L6-v2` + FAISS semantic retrieval.
4. **Week 6:** local cross-encoder reranking.
5. **Week 7:** dynamic few-shot selection and held-out evaluation discipline.


## Architecture and scope

```text
existing transcript
      |
      v
MiniLM + FAISS retrieval
      |
      v
cross-encoder reranking
      |
      +---- optional dynamic examples
      |
      v
Gemini structured extraction
      |
      v
deterministic validation
      |
      v
operational brief + evaluation
```

The project intentionally excludes transcription, Teams/Zoom integration, task or email automation, fine-tuning, a web UI, deployment, and multi-agent orchestration. Gemini handles ambiguous interpretation and synthesis; deterministic/local components handle retrieval, reranking, validation, metrics, and the targeted mitigation.


In [ ]:
# If needed in Colab: %pip install -q sentence-transformers faiss-cpu openai pandas numpy
import os, sys, json, math, pathlib
import pandas as pd
import numpy as np

# Run this notebook from the repository's project/ directory.
assert pathlib.Path('capstone_core.py').exists(), 'Run from project/ so capstone_core.py and data/ are available.'
from capstone_core import *

print('Held-out cases:', len(EVAL_CASES))
print('Gemini model:', GEMINI_MODEL)
print('Embedding model:', EMBED_MODEL)
print('Reranker:', RERANK_MODEL)
print('Local comparison model:', LOCAL_MODEL)
assert 10 <= len(EVAL_CASES) <= 20


## Ablation evaluation

The same 12 held-out cases are evaluated under three configurations:

- **A — transcript only**
- **B — transcript + retrieved/reranked operational context**
- **C — transcript + retrieved/reranked context + dynamic few-shot examples**

Concrete metrics are decision precision/recall/F1, action-item precision/recall/F1, owner accuracy, blocker F1, follow-up-question F1, structured-output validity, evidence support, and latency. The model is instructed to include transcript evidence for every extracted operational item.


In [ ]:
if not os.getenv('GEMINI_API_KEY'):
    print('API-BLOCKED: set GEMINI_API_KEY, then rerun this cell. No synthetic model results will be substituted.')
    ABLATION_RESULTS, METRICS = {}, pd.DataFrame()
else:
    ABLATION_RESULTS, METRICS = run_ablation()
    display(METRICS.round(3))
    metric_cols = ['decision_precision','decision_recall','decision_f1','action_precision','action_recall','action_f1','owner_accuracy','blocker_f1','question_f1','valid','evidence_support','latency_s']
    ABLATION_SUMMARY = METRICS.groupby('config')[metric_cols].mean().round(3)
    display(ABLATION_SUMMARY)


## LLM-as-judge: 12 held-out cases × 3 named dimensions

Gemini separately judges each candidate brief on a 1–5 scale for exactly three dimensions required for a defensible qualitative pass:

1. **correctness** — are the operational outcomes right?
2. **grounding** — are material claims supported by the transcript?
3. **operational usefulness** — is the result concise and usable after the meeting?


In [ ]:
if not ABLATION_RESULTS:
    print('Run the ablation first.')
else:
    JUDGE_DF = run_judge(ABLATION_RESULTS)
    display(JUDGE_DF)
    JUDGE_SUMMARY = JUDGE_DF.groupby('config')[['correctness','grounding','operational_usefulness']].mean().round(3)
    display(JUDGE_SUMMARY)


## Required failure mode and component trace

The declared failure is a suggestion/discussion point being incorrectly promoted to a confirmed decision. The trace prefers a case where dynamic few-shot context causes a decision-F1 regression relative to retrieved context alone. If that is absent, it checks whether retrieval caused the regression; otherwise it attributes the lowest-scoring case to generation rather than inventing a retrieval failure. Retrieved rules/examples and the A/B/C outputs provide the evidence.


In [ ]:
if not ABLATION_RESULTS:
    FAILURE = None
    print('Run the ablation first.')
else:
    FAILURE = trace_failure(ABLATION_RESULTS)
    cid = FAILURE['case_id']
    print('Failure case:', cid)
    print('Responsible component:', FAILURE['component'])
    print('\nTranscript:\n', FAILURE['case']['transcript'])
    print('\nGold:\n', json.dumps(FAILURE['case']['gold'], indent=2))
    for cfg in ['A','B','C']:
        print(f'\n--- {cfg} ---')
        print(json.dumps(ABLATION_RESULTS[cfg][cid]['brief'], indent=2))
        print('scores:', FAILURE['scores'][cfg][cid])
        if cfg in ('B','C'):
            print('retrieved rules:')
            for x in ABLATION_RESULTS[cfg][cid]['retrieval']['rules']:
                print(x['doc']['id'], round(x['rerank_score'],3), x['doc']['text'])
        if cfg == 'C':
            print('retrieved examples:')
            for x in ABLATION_RESULTS[cfg][cid]['retrieval']['examples']:
                print(x['doc']['label'], round(x['rerank_score'],3), x['doc']['text'])


## Mitigation

The targeted guardrail rejects a predicted decision when its evidence contains suggestion/hedging language but no commitment cue. It is intentionally deterministic and narrow: it mitigates the known error without replacing Gemini's reasoning. The cell below measures whether it improves decision F1 or harms recall.


In [ ]:
if ABLATION_RESULTS:
    rows=[]
    for case in EVAL_CASES:
        raw=ABLATION_RESULTS['C'][case['id']]
        brief,removed=mitigate_decisions(raw['brief']) if raw.get('valid') else ({},[])
        r=dict(raw); r['brief']=brief; r['valid'],r['validation_errors']=validate_brief(brief) if brief else (False,['invalid'])
        s=score_case(case,r); s['removed_decisions']=len(removed); rows.append(s)
    MITIGATION_DF=pd.DataFrame(rows)
    display(MITIGATION_DF.round(3))
    print('Configuration C mean decision F1:', round(METRICS[METRICS.config=='C'].decision_f1.mean(),3))
    print('Mitigated mean decision F1:', round(MITIGATION_DF.decision_f1.mean(),3))


## Small local model comparison

Qwen3.5-0.8B is tested only on the narrow span-classification problem (`decision`, `action_item`, `blocker`, `follow_up_question`, `context`). This measures whether a small local model is good enough for a limited role without forcing it to perform the full reasoning/synthesis task. Start a llama.cpp OpenAI-compatible server, set `RUN_LOCAL_MODEL=1`, and run the cell.


In [ ]:
if os.getenv('RUN_LOCAL_MODEL','0') != '1':
    print('Local comparison skipped. Set RUN_LOCAL_MODEL=1 when llama.cpp is running.')
else:
    LOCAL_DF=run_local_comparison()
    display(LOCAL_DF)
    print('Local classification accuracy:', round(LOCAL_DF.correct.mean(),3))
    print('Mean local latency:', round(LOCAL_DF.latency_s.mean(),3), 'seconds')


## Final selection and submission guard

The final architecture should use the simplest configuration supported by the measurements. Do not keep dynamic few-shot prompting merely because it is more sophisticated. Before submission, the notebook must contain the measured A/B/C tables, the 12-case judge pass, the failure trace, and the mitigation comparison. Save the executed notebook and export it to HTML.


In [ ]:
problems=[]
if not ABLATION_RESULTS: problems.append('Ablation results missing')
if 'JUDGE_DF' not in globals() or len(JUDGE_DF) < len(EVAL_CASES): problems.append('LLM-as-judge results missing/incomplete')
if FAILURE is None: problems.append('Failure trace missing')
if problems:
    print('NOT READY FOR SUBMISSION')
    for p in problems: print('-',p)
else:
    print('READY FOR SUBMISSION')
    print('- 12 held-out cases evaluated under A/B/C')
    print('- deterministic metrics reported')
    print('- judge completed on correctness, grounding, operational usefulness')
    print('- failure traced to a component with ablation/log evidence')
    print('- mitigation measured')

print('\nExport after execution:')
print('jupyter nbconvert --to html week8_meeting_to_action_capstone.ipynb')
